# 主動型投資- 資料處理
**以「週」為單位做台股的買賣，賺價差**
流程 (網路爬蟲都使用 `WeekTermInvestment_DataScrape` 的工具)：
1. 輸入台股代碼 (ABBR)
2. 檢查 DataCollection/ABBR 資料夾內的 5 份 CSV (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)
3. 沒有資料夾或檔案有誤時：先用 `WeekTermInvestment_DataScrape` 抓取新資料，抓取成功後再刪除 DataCollection/ABBR 資料夾內的舊資料 (若有的話)，再存入新資料
4. 設定分析用的週次變數

投資操作：
* 買_時機點：大戶籌碼比例增加 & 大戶借券量下降 & 融資(韭菜)餘額下降，買進價格<=大戶平均成本
* 賣_時機點：大戶籌碼比例下降 & 大戶借券量增加 & 融資(韭菜)餘額上升 & 融資維持率持平/下降，賣出價格>=韭菜平均成本

### 匯入Package

In [ ]:
import ast
import builtins
import json
import re
import types
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()  # 本 notebook 所在的資料夾
DATASCRAPE_NOTEBOOK = BASE_DIR / "WeekTermInvestment_DataScrape.ipynb"


def _is_definition(node, namespace):
    """只保留 import、函式/類別定義、常數設定。
    常數設定的等號右邊不能呼叫函式 (以免執行到 input() 或爬蟲)，用到的變數也必須已經定義
    (例如執行格中的 outputs = [(df_inst, ...)] 會被略過，因為 df_inst 是爬蟲的結果)"""
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return True
    if isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
        sub_nodes = list(ast.walk(node.value))
        names = [n.id for n in sub_nodes if isinstance(n, ast.Name)]
        return not any(isinstance(n, ast.Call) for n in sub_nodes) and all(
            name in namespace or hasattr(builtins, name) for name in names
        )
    return False


def load_datascrape_tools(nb_path):
    """把 WeekTermInvestment_DataScrape.ipynb 當成 package 載入：
    只執行裡面的 import 與函式/常數定義，不會執行「輸入代碼 → 爬資料」那一格"""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    module = types.ModuleType(Path(nb_path).stem)
    module.__file__ = str(nb_path)
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        try:
            tree = ast.parse("".join(cell["source"]))
        except SyntaxError:  # 例如含有 %magic 的格子，不是工具定義，略過
            continue
        # 逐行判斷、逐行執行，才知道某個變數在這一行之前是否已經定義
        for node in tree.body:
            if _is_definition(node, module.__dict__):
                exec(compile(ast.Module(body=[node], type_ignores=[]), str(nb_path), "exec"), module.__dict__)
    return module


# 所有網路爬蟲 (籌碼資料、週K線股價) 都在 DataScrape 中
ds = load_datascrape_tools(DATASCRAPE_NOTEBOOK)
DATA_DIR = BASE_DIR / ds.DATA_DIR  # 各股票的資料夾 (ABBR) 都放在這裡 (DataCollection，定義在 DataScrape)
print(f"已載入 {DATASCRAPE_NOTEBOOK.name} 的工具")

### 函式定義(週資料檔案檢查)

In [ ]:
# ===== 週資料：檢查 ABBR 資料夾內的 5 份 CSV，沒有或有誤就用 DataScrape 重新取得 =====
# 5 份資料的檔名標籤與欄位定義在 DataScrape 的 WEEK_FILE_COLUMNS
# (法人買賣超、資券餘額、借券賣出餘額、大戶籌碼、股價週K線)


def _is_yyyymmdd(s):
    try:
        datetime.strptime(s, "%Y%m%d")
        return True
    except ValueError:
        return False


def _latest_week_file(folder, abbr, file_tag):
    """回傳資料夾內 ABBR_{file_tag}_YYYYMMDD.csv 中日期最新的那份，沒有就回傳 None"""
    pattern = re.compile(rf"{abbr}_{file_tag}_(\d{{8}})\.csv")
    matches = [f for f in Path(folder).glob("*.csv") if (m := pattern.fullmatch(f.name)) and _is_yyyymmdd(m.group(1))]
    return max(matches, key=lambda f: f.name, default=None)


def check_week_files(folder, abbr):
    """檢查資料夾內的 5 份週資料 CSV，回傳問題清單 (空清單代表檔案都正確)
    檢查項目：至少 5 份 CSV、5 種檔名都有 (結尾是合法的年月日)、檔案讀得到且欄位數正確、有資料"""
    n_csv = len(list(Path(folder).glob("*.csv")))
    problems = []
    if n_csv < len(ds.WEEK_FILE_COLUMNS):
        problems.append(f"CSV 檔只有 {n_csv} 份，至少要 {len(ds.WEEK_FILE_COLUMNS)} 份")
    for tag, columns in ds.WEEK_FILE_COLUMNS.items():
        latest = _latest_week_file(folder, abbr, tag)  # 同一種資料有多份時，檢查日期最新的那份
        if latest is None:
            problems.append(f"缺少 {abbr}_{tag}_檔案建立年月日.csv")
            continue
        try:
            df = pd.read_csv(latest, encoding="utf-8-sig")
        except Exception as e:
            problems.append(f"{latest.name} 無法讀取：{e}")
            continue
        if df.shape[1] != len(columns):
            problems.append(f"{latest.name} 有 {df.shape[1]} 欄，應該是 {len(columns)} 欄")
        elif df.empty:
            problems.append(f"{latest.name} 沒有資料")
    return problems


def ensure_week_data(abbr):
    """確保 ABBR 資料夾內有正確的 5 份週資料 CSV，回傳資料夾路徑
    沒有資料夾或檔案有誤時：先用 DataScrape 抓取 5 份新資料，全部成功後才刪除資料夾內舊的 CSV (若有的話)，
    再存入新資料；抓取失敗時舊檔都不會被刪除"""
    folder = DATA_DIR / abbr
    if not folder.is_dir():
        print(f"⚠ 找不到資料夾「{abbr}」")
    else:
        problems = check_week_files(folder, abbr)
        if not problems:
            print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確")
            return folder
        print(f"⚠ 資料夾「{abbr}」內的檔案有誤：")
        for p in problems:
            print(f"   - {p}")

    print("使用 WeekTermInvestment_DataScrape 抓取新資料，抓取成功後再刪除舊的 CSV 檔…")
    week_data = ds.fetch_all_week_data(abbr)
    for old in sorted(folder.glob("*.csv")):  # 資料夾不存在時沒有舊檔
        old.unlink()
        print(f"已刪除舊檔：{old.name}")
    for file_tag, df in week_data.items():
        csv_path = ds.save_week_csv(df, abbr, file_tag, base_dir=DATA_DIR)
        print(f"{df.attrs['symbol']} {file_tag} 共 {len(df)} 筆，已存檔：{csv_path}")

    problems = check_week_files(folder, abbr)
    if problems:
        raise RuntimeError(f"重新取得資料後，資料夾「{abbr}」內的檔案仍有誤：\n" + "\n".join(problems))
    print(f"✔ 資料夾「{abbr}」內的 {len(ds.WEEK_FILE_COLUMNS)} 份 CSV 檔都正確")
    return folder

### 函式定義(大戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆法人買賣超週資料，外資、投信+自營商 各自計算期間內的買進均價：
* 賣超不影響成本，只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時為 NaN
* 當週典型價 = (高 + 低 + 收) / 3，來自 `ABBR_Stock_Price_Week`，與法人資料用 ISO 年週對齊
* 投信、自營商視為同一個大戶，同一週的買賣超先相加，再判斷是買超或賣超
* 期初持股與成本無法得知，所以這是大戶「期間內新買進部位」的平均成本

In [ ]:
# ===== 平均成本：共用函式 + 大戶（外資、投信+自營商）買入平均成本 =====
# 週次一律用 ISO 年週 (年, 第幾週)：週一到週日為一週，與 date.isocalendar() 相同，
# 例如 date(2026, 11, 6).isocalendar()[:2] == (2026, 45)
# 各份 CSV 同一週的日期不一定相同 (例如法人用週五 11/6、資券餘額用週一 11/2)，所以用 ISO 年週對齊

# 大戶分組: 法人買賣超 CSV 中要加總的欄位 (投信+自營商 視為同一個大戶，同一週先相加)
INSTITUTION_GROUPS = {"外資": ["外資(張)"], "投信+自營商": ["投信(張)", "自營商(張)"]}
INSTITUTIONAL_FILE_TAG = "Institutional_Trading_Week_8cols"


def _to_number(col):
    """CSV 中的數字可能是有千分位逗號或百分比的字串 (例如 -2,927、84.17%)，轉成數值 (百分比只去掉 % 符號)；
    無法轉換的 (例如「-」) 變成 NaN；已經是數值的欄位直接回傳"""
    if pd.api.types.is_numeric_dtype(col):
        return col
    return pd.to_numeric(col.astype(str).str.replace(",", "").str.rstrip("%"), errors="coerce")


def read_week_csv(abbr, file_tag, data_dir=None):
    """讀取 data_dir/ABBR 資料夾中日期最新的 ABBR_{file_tag}_YYYYMMDD.csv (data_dir 預設為 DATA_DIR)，
    依日期由新到舊排序，並加上「年週」欄位 (ISO 年, 第幾週)"""
    path = _latest_week_file(Path(data_dir or DATA_DIR) / abbr, abbr, file_tag)
    if path is None:
        raise FileNotFoundError(f"資料夾「{abbr}」中找不到 {abbr}_{file_tag}_檔案建立年月日.csv")
    df = pd.read_csv(path, encoding="utf-8-sig")
    df["_date"] = pd.to_datetime(df.iloc[:, 0], format="%Y/%m/%d")  # 第一欄是日期
    df = df.sort_values("_date", ascending=False).reset_index(drop=True)
    df["年週"] = [d.isocalendar()[:2] for d in df.pop("_date")]
    return df


def _week_window(df, pivot_year_week, input_week_num, label):
    """從 read_week_csv 讀到的週資料 (由新到舊) 中，取出以 pivot_year_week 為終點 (包含) 往前 input_week_num 筆"""
    pivot_year_week = tuple(pivot_year_week)
    positions = [i for i, yw in enumerate(df["年週"]) if yw == pivot_year_week]
    if not positions:
        raise ValueError(
            f"{label}資料中沒有 {pivot_year_week} 這一週 (資料範圍 {df['年週'].iloc[-1]} ~ {df['年週'].iloc[0]})"
        )
    window = df.iloc[positions[0] : positions[0] + input_week_num]
    if len(window) < input_week_num:
        raise ValueError(f"{pivot_year_week} 往前只有 {len(window)} 筆{label}資料，不足 {input_week_num} 筆")
    return window


def _week_typical_price(price, window):
    """回傳 window 中每一週的典型價 (高 + 低 + 收) / 3 (比只用收盤價更接近整週的成交價位)，index 與 window 相同
    price 是 read_week_csv 讀到的股價週資料"""
    typical = (_to_number(price["高"]) + _to_number(price["低"]) + _to_number(price["收"])) / 3
    price_by_week = dict(zip(price["年週"], typical))
    missing = [yw for yw in window["年週"] if pd.isna(price_by_week.get(yw))]
    if missing:
        raise ValueError(f"股價資料中沒有這些週的高/低/收：{missing}")
    return pd.Series([price_by_week[yw] for yw in window["年週"]], index=window.index)


def _buy_avg_cost(window, lots, week_price, label):
    """賣出不影響成本，只用買進 (張數 > 0) 的週：平均成本 = Σ(買進張數 × 當週典型價) / Σ(買進張數)
    期初持股與成本無法得知，所以這是「期間內新買進部位」的平均成本；期間內都沒有買進時是 NaN"""
    if lots.isna().any():
        raise ValueError(f"{label} 的張數有無法轉成數字的值，年週：{window.loc[lots.isna(), '年週'].tolist()}")
    buys = lots > 0
    total_lots = lots[buys].sum()
    total_money = (lots[buys] * week_price[buys]).sum()  # 張 × 元；1 張 = 1000 股，相除時單位抵銷，即每股價格
    return total_money / total_lots if total_lots > 0 else float("nan")


def _institutional_avg_cost(inst, price, pivot_year_week, input_week_num):
    """用已讀進來的法人買賣超、股價週資料計算大戶平均成本 (calc_institutional_avg_cost 的計算本體)"""
    window = _week_window(inst, pivot_year_week, input_week_num, "法人買賣超")
    week_price = _week_typical_price(price, window)
    return {
        name: _buy_avg_cost(window, sum(_to_number(window[c]) for c in columns), week_price, name)
        for name, columns in INSTITUTION_GROUPS.items()
    }


def calc_institutional_avg_cost(abbr, pivot_year_week, input_week_num, data_dir=None):
    """計算大戶 (外資、投信+自營商) 在期間內的買進均價，回傳 {"外資": 平均成本, "投信+自營商": 平均成本}
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆法人買賣超週資料
    - 只用買超的週：平均成本 = Σ(買超張數 × 當週典型價) / Σ(買超張數)；期間內都沒有買超時是 NaN"""
    inst = read_week_csv(abbr, INSTITUTIONAL_FILE_TAG, data_dir)
    price = read_week_csv(abbr, ds.PRICE_FILE_TAG, data_dir)
    return _institutional_avg_cost(inst, price, pivot_year_week, input_week_num)

### 函式定義(散戶買入平均成本)
以 `PIVOT_YEAR_WEEK` 為錨點(包含)，往前 `INPUT_WEEK_NUM` 筆資券餘額週資料，算法比照大戶：
* 融資減少不影響成本，只用融資增加的週：平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時為 NaN
* 融資增減取自 `ABBR_Margin_Week_11cols` 的「融資 → 增減」；當週典型價 = (高 + 低 + 收) / 3，與資券資料用 ISO 年週對齊
* 期初融資部位與成本無法得知，所以這是散戶「期間內新增融資部位」的平均成本

In [ ]:
# ===== 散戶（融資）買入平均成本 =====

MARGIN_FILE_TAG = "Margin_Week_11cols"
# 資券餘額 CSV 有兩個「增減」欄：第一個是融資、第二個是融券 (pandas 讀進來會改名為「增減.1」)
MARGIN_CHANGE_COLUMN = "增減"


def calc_retail_avg_cost(abbr, pivot_year_week, input_week_num, data_dir=None):
    """計算散戶 (融資) 在期間內的買進均價，回傳平均成本
    - 以 pivot_year_week (ISO 年, 第幾週) 為終點 (包含)，往前取 input_week_num 筆資券餘額週資料
    - 融資減少 (賣出還款) 不影響成本，只用融資增加的週：
      平均成本 = Σ(融資增加張數 × 當週典型價) / Σ(融資增加張數)；期間內融資都沒有增加時是 NaN"""
    margin = read_week_csv(abbr, MARGIN_FILE_TAG, data_dir)
    price = read_week_csv(abbr, ds.PRICE_FILE_TAG, data_dir)
    window = _week_window(margin, pivot_year_week, input_week_num, "資券餘額")
    lots = _to_number(window[MARGIN_CHANGE_COLUMN])  # 每週融資增減張數
    return _buy_avg_cost(window, lots, _week_typical_price(price, window), "融資")

### 函式定義(假設驗證：買進訊號回測)
假設：以錨點往前 `INPUT_WEEK_NUM` 筆資料做線性回歸，當
* `Major_Holders_Week` 的大戶籌碼比例斜率 > 0 (增加)
* `Securities_Lending_and_Borrowing_Week` 的借券賣出餘額斜率 < 0 (減少)
* `Margin_Week` 的融資餘額斜率 < 0 (減少)

時適合買進。驗證錨點之後 `OUTPUT_WEEK_NUM` 週的股價 (`Stock_Price_Week` 的最低價) 與同期間大戶買入平均成本的關係 (命中的定義見 `HIT_RULES`)，並與沒有訊號的樣本比較，看訊號是否真的有幫助。

In [ ]:
# ===== 假設驗證（回測）：買進訊號出現後，未來 output_week_num 週的股價與大戶平均成本的關係 =====
# 買進訊號：錨點往前 input_week_num 筆資料做線性回歸，
#   大戶籌碼比例斜率 > 0、借券賣出餘額斜率 < 0、融資餘額斜率 < 0
# 驗證 (命中的定義見 HIT_RULES)：錨點之後 (不包含) output_week_num 週的最低價與大戶平均成本比較

MAJOR_HOLDERS_FILE_TAG = "Major_Holders_Week_5cols"
LENDING_FILE_TAG = "Securities_Lending_and_Borrowing_Week_8cols"
# 訊號名稱: (檔名標籤, 欄位, 斜率的方向 +1 / -1)
# 借券賣出餘額是借券 CSV 的第二個「餘額」欄 (pandas 改名為「餘額.1」)；融資餘額是資券餘額 CSV 的第一個「餘額」欄
SIGNAL_TRENDS = {
    "大戶籌碼": (MAJOR_HOLDERS_FILE_TAG, "大戶籌碼", 1),
    "借券賣出餘額": (LENDING_FILE_TAG, "餘額.1", -1),
    "融資餘額": (MARGIN_FILE_TAG, "餘額", -1),
}
MIN_TREND_POINTS = 3  # 線性回歸至少要有幾週的數值

# 命中的定義: (說明, 用 未來最低價 與 成本 判斷是否命中)
HIT_RULES = {
    "touch": ("未來最低價 <= 大戶成本：有機會用 <= 成本的價格買進", lambda low, cost: low <= cost),
    "hold": ("未來最低價 >= 大戶成本：股價守住大戶成本 (成本是支撐)", lambda low, cost: low >= cost),
}
DISTANCE_GROUPS = 5  # 「同錨點同距離差」把 錨點收盤價 / 成本 - 1 分成幾組 (依分位數)


def load_stock_week_data(abbr, data_dir=None):
    """讀取一檔股票的 5 份週資料，回傳 {檔名標籤: DataFrame}；
    缺少任何一份、或檔案讀不到 (例如正在寫入) 就回傳 None，不會觸發 DataScrape
    除了日期、年週以外的欄位都先轉成數值，之後計算就不用每次重新轉換"""
    folder = Path(data_dir or DATA_DIR) / abbr
    if any(_latest_week_file(folder, abbr, tag) is None for tag in ds.WEEK_FILE_COLUMNS):
        return None
    data = {}
    for tag in ds.WEEK_FILE_COLUMNS:
        try:
            df = read_week_csv(abbr, tag, data_dir)
        except ValueError:
            return None
        for col in df.columns[1:-1]:  # 第一欄是日期、最後一欄是年週
            df[col] = _to_number(df[col])
        data[tag] = df
    return data


def _first_unfinished_week(price_path):
    """抓資料當週如果還沒收盤，那一週的週K線不完整，不能當作「未來」的完整一週。
    回傳第一個不完整的年週：週一 ~ 週五抓的是當週；週六、週日抓的 (當週已收盤) 是下一週"""
    created = datetime.strptime(Path(price_path).stem[-8:], "%Y%m%d")
    if created.weekday() >= 5:
        created += timedelta(days=7)
    return created.isocalendar()[:2]


def _trend_slope(values):
    """線性回歸斜率 (values 由新到舊，先反轉成由舊到新)
    某幾週沒有數值時 (例如大戶籌碼在 2026/7/9 那週是「-」)，用其餘的週計算，x 仍是原本的週次位置；
    有數值的週少於 MIN_TREND_POINTS 時回傳 NaN"""
    y = _to_number(values).to_numpy(dtype=float)[::-1]
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    if valid.sum() < MIN_TREND_POINTS:
        return float("nan")
    return np.polyfit(x[valid], y[valid], 1)[0]


def buy_signal_slopes(data, pivot_year_week, input_week_num):
    """回傳錨點往前 input_week_num 筆資料中，各訊號欄位的線性回歸斜率 {訊號名稱: 斜率}"""
    return {
        name: _trend_slope(_week_window(data[tag], pivot_year_week, input_week_num, name)[column])
        for name, (tag, column, _) in SIGNAL_TRENDS.items()
    }


def is_buy_signal(slopes):
    """大戶籌碼斜率 > 0、借券賣出餘額斜率 < 0、融資餘額斜率 < 0 時為買進訊號"""
    return all(np.sign(slopes[name]) == direction for name, (_, _, direction) in SIGNAL_TRENDS.items())


def backtest_buy_signal(input_week_nums, output_week_num, data_dir=None):
    """對 data_dir 中每一檔股票、每個 input_week_num、每個可用的錨點做回測，回傳 DataFrame (每列一個樣本)
    - 錨點要讓 5 份資料都有往前 input_week_num 筆，且之後有 output_week_num 週已結束的股價
    - 缺少 5 份 csv 的股票會跳過；只讀取現有的 csv，不會觸發 DataScrape"""
    data_dir = Path(data_dir or DATA_DIR)
    rows, incomplete, errors = [], [], []
    for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        abbr = folder.name
        data = load_stock_week_data(abbr, data_dir)
        if data is None:
            incomplete.append(abbr)
            continue
        price = data[ds.PRICE_FILE_TAG]
        weeks = {tag: list(df["年週"]) for tag, df in data.items()}  # 各份資料的年週 (由新到舊)
        current_week = _first_unfinished_week(_latest_week_file(folder, abbr, ds.PRICE_FILE_TAG))
        for n in input_week_nums:
            # 錨點必須出現在每一份資料中，所以只需要從其中一份 (資券餘額，筆數最少) 的年週裡找
            for pivot in weeks[MARGIN_FILE_TAG]:
                if not all(pivot in w and w.index(pivot) + n <= len(w) for w in weeks.values()):
                    continue
                p = weeks[ds.PRICE_FILE_TAG].index(pivot)
                future = price.iloc[max(p - output_week_num, 0) : p]  # 錨點之後的週 (由新到舊)
                if len(future) < output_week_num or future["年週"].iloc[0] >= current_week:
                    continue
                try:
                    slopes = buy_signal_slopes(data, pivot, n)
                    cost = _institutional_avg_cost(data[INSTITUTIONAL_FILE_TAG], price, pivot, n)
                except ValueError as e:  # 這個錨點的資料有問題 (例如某週沒有股價)，只跳過這個樣本
                    errors.append((abbr, n, pivot, str(e)))
                    continue
                rows.append({
                    "股票": abbr, "INPUT_WEEK_NUM": n, "錨點": pivot,
                    **{f"斜率_{k}": v for k, v in slopes.items()},
                    "訊號": is_buy_signal(slopes),
                    **{f"成本_{k}": v for k, v in cost.items()},
                    "錨點收盤": price["收"].iloc[p],
                    "未來最低": future["低"].min(),
                    "未來最高": future["高"].max(),
                    "未來收盤": future["收"].iloc[0],  # 未來最後一週的收盤價
                })
    print(f"回測股票 {len({r['股票'] for r in rows})} 檔，樣本 {len(rows)} 筆；跳過資料不完整的股票 {len(incomplete)} 檔、"
          f"資料有誤的樣本 {len(errors)} 筆 (分布在 {len({e[0] for e in errors})} 檔)")
    return pd.DataFrame(rows)


def _signal_diff(d, column, strata):
    """在 strata 欄位相同的每一組樣本內，計算 有訊號 - 沒訊號 的 column 平均，再依訊號數加權平均
    (例如 strata=["錨點"] 就是在同一段時間內比較，排除大盤漲跌的影響)"""
    diffs = [
        (p.loc[p["訊號"], column].mean() - p.loc[~p["訊號"], column].mean(), p["訊號"].sum())
        for _, p in d.groupby(strata) if p["訊號"].any() and not p["訊號"].all()
    ]
    return np.average([x for x, _ in diffs], weights=[w for _, w in diffs]) if diffs else np.nan


def summarize_backtest(results, hit_rule="touch"):
    """依 INPUT_WEEK_NUM 彙整回測結果，比較「有訊號」與「沒有訊號」的樣本 (只計算平均成本不是 NaN 的樣本)
    hit_rule 是 HIT_RULES 的 key：
    - 命中率：依 hit_rule 判斷未來 output_week_num 週的最低價與大戶平均成本
    - 同錨點差：每個錨點 (同一段時間) 先算 有訊號 - 沒訊號 的命中率，再依訊號數加權平均，排除大盤漲跌的影響
    - 同錨點同距離差：再把 錨點收盤價 / 成本 - 1 (股價離成本多遠) 分成 DISTANCE_GROUPS 組，同一組內比較；
      股價離成本越遠，本來就越不容易跌回成本，這樣才能看出命中率的差異是不是訊號造成的
    - 4週報酬差：錨點收盤價買進、持有到未來最後一週收盤的報酬，同錨點比較 有訊號 - 沒訊號
    - (touch) 需下跌命中率：錨點收盤價 > 成本 (要等股價跌下來才買得到) 的樣本中的命中率
    - (touch) 買進後報酬：命中時以成本價買進，到未來最後一週收盤的平均報酬"""
    is_hit = HIT_RULES[hit_rule][1]
    rows = []
    for n, g in results.groupby("INPUT_WEEK_NUM"):
        row = {"INPUT_WEEK_NUM": n, "樣本數": len(g), "錨點數": g["錨點"].nunique(), "訊號數": int(g["訊號"].sum())}
        for name in INSTITUTION_GROUPS:
            cost = f"成本_{name}"
            d = g[g[cost].notna()].copy()
            d["命中"] = is_hit(d["未來最低"], d[cost])
            d["距離"] = d["錨點收盤"] / d[cost] - 1
            d["距離分組"] = pd.qcut(d["距離"], DISTANCE_GROUPS, labels=False, duplicates="drop")
            d["4週報酬"] = d["未來收盤"] / d["錨點收盤"] - 1
            sig, none = d[d["訊號"]], d[~d["訊號"]]
            row[f"{name}_訊號命中率"] = sig["命中"].mean()
            row[f"{name}_無訊號命中率"] = none["命中"].mean()
            row[f"{name}_同錨點差"] = _signal_diff(d, "命中", ["錨點"])
            row[f"{name}_同錨點同距離差"] = _signal_diff(d, "命中", ["錨點", "距離分組"])
            row[f"{name}_訊號距離中位數"] = sig["距離"].median()
            row[f"{name}_無訊號距離中位數"] = none["距離"].median()
            row[f"{name}_4週報酬差"] = _signal_diff(d, "4週報酬", ["錨點"])
            if hit_rule == "touch":
                d["報酬"] = (d["未來收盤"] / d[cost] - 1).where(d["命中"])
                sig, none = d[d["訊號"]], d[~d["訊號"]]
                row[f"{name}_訊號需下跌命中率"] = sig.loc[sig["錨點收盤"] > sig[cost], "命中"].mean()
                row[f"{name}_無訊號需下跌命中率"] = none.loc[none["錨點收盤"] > none[cost], "命中"].mean()
                row[f"{name}_訊號買進後報酬"] = sig["報酬"].mean()
                row[f"{name}_無訊號買進後報酬"] = none["報酬"].mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("INPUT_WEEK_NUM")

### 函式定義(來回交易回測：買得比大戶成本低、賣得比散戶成本高)
* 前提：錨點時 大戶成本 < 散戶成本，且價差 (散戶成本 / 大戶成本 - 1) >= `ROUND_TRIP_MIN_SPREAD` (3%)，成功時至少賺 3%
* 買進：錨點之後 `OUTPUT_WEEK_NUM` 週內，股價碰到大戶成本就買 (價格 <= 大戶成本)
* 賣出：買進之後 `OUTPUT_WEEK_NUM` 週內，股價碰到散戶成本就賣 (價格 >= 散戶成本)；沒碰到就在最後一週收盤賣出
* `ROUND_TRIP_PATTERNS`：提高來回成功率的模式。關鍵是「股價現在就在大戶成本附近」以及「價差相對於這檔股票平常一週的波動夠小」

In [ ]:
# ===== 來回交易回測：買進價格 <= 大戶成本、之後賣出價格 >= 散戶成本 =====
# 前提：錨點時 大戶成本 < 散戶成本 (才有價差可賺)
# 買進：錨點之後 buy_weeks 週內，最低價碰到大戶成本就買 (當週開盤已低於成本時用開盤價)
# 賣出：買進之後 hold_weeks 週內，最高價碰到散戶成本就賣 (當週開盤已高於成本時用開盤價)；
#       沒碰到就在第 hold_weeks 週收盤賣出

ROUND_TRIP_COSTS = {"外資": "外資", "投信自營": "投信+自營商"}  # 回測用的大戶成本: INSTITUTION_GROUPS 的 key


def _window_change(values, percent_points):
    """window 期間 (由新到舊) 用線性回歸算出的變化量：
    percent_points=True 用於百分比欄位，回傳變化幾個百分點；否則回傳相對平均值的變化比例"""
    y = _to_number(values).to_numpy(dtype=float)[::-1]
    x = np.arange(len(y))
    valid = ~np.isnan(y)
    if valid.sum() < MIN_TREND_POINTS:
        return np.nan
    change = np.polyfit(x[valid], y[valid], 1)[0] * (len(y) - 1)
    if percent_points:
        return change
    mean = np.abs(y[valid]).mean()
    return change / mean if mean else np.nan


def simulate_round_trip(future, buy_price, sell_price, buy_weeks, hold_weeks):
    """future 是錨點之後的股價週資料 (由舊到新，至少 buy_weeks + hold_weeks 週)
    回傳 (是否買到, 是否賣到, 買價, 賣價, 錨點後第幾週買到, 錨點後第幾週賣出)"""
    for i in range(buy_weeks):
        week = future.iloc[i]
        if week["低"] <= buy_price:
            fill = min(buy_price, week["開"])
            for j in range(i + 1, i + 1 + hold_weeks):  # 買進的下一週才開始賣
                if future.iloc[j]["高"] >= sell_price:
                    return True, True, fill, max(sell_price, future.iloc[j]["開"]), i + 1, j + 1
            return True, False, fill, future.iloc[i + hold_weeks]["收"], i + 1, i + 1 + hold_weeks
    return False, False, np.nan, np.nan, np.nan, np.nan


def round_trip_features(data, pivot_year_week, input_week_num):
    """錨點往前 input_week_num 筆資料的籌碼、股價特徵 (f_ 開頭)：
    百分比欄位是變化幾個百分點，其他是相對變化比例，買超強度 = 期間買超張數 / 成交量，
    週波動率 = 每週 (最高 - 最低) / 收盤 的平均"""
    w = {tag: _week_window(df, pivot_year_week, input_week_num, tag) for tag, df in data.items()}
    inst, major, margin, prices = w[INSTITUTIONAL_FILE_TAG], w[MAJOR_HOLDERS_FILE_TAG], w[MARGIN_FILE_TAG], w[ds.PRICE_FILE_TAG]
    close = prices["收"]
    volume = inst["成交量"].sum()
    return {
        "f_大戶籌碼": _window_change(major["大戶籌碼"], True),
        "f_外資持股": _window_change(major["外資籌碼"], True),
        "f_董監持股": _window_change(major["董監持股"], True),
        "f_借券賣出餘額": _window_change(w[LENDING_FILE_TAG]["餘額.1"], False),
        "f_融資餘額": _window_change(margin["餘額"], False),
        "f_融資使用率": _window_change(margin["使用率％"], True),
        "f_融券餘額": _window_change(margin["餘額.1"], False),
        "f_券資比": _window_change(margin["券資比"], True),
        "f_外資買超強度": inst["外資(張)"].sum() / volume if volume else np.nan,
        "f_投信自營買超強度": (inst["投信(張)"] + inst["自營商(張)"]).sum() / volume if volume else np.nan,
        "f_股價動能": close.iloc[0] / close.iloc[-1] - 1,
        "f_近4週動能": close.iloc[0] / close.iloc[min(3, len(close) - 1)] - 1,
        "f_成交量變化": _window_change(inst["成交量"], False),
        "f_週波動率": ((prices["高"] - prices["低"]) / close).mean(),
    }


def backtest_round_trip(input_week_num, buy_weeks, hold_weeks, data_dir=None):
    """對 data_dir 中每一檔股票、每個可用的錨點，計算大戶/散戶成本、特徵與來回交易結果，回傳 DataFrame (每列一個樣本)
    錨點之後要有 buy_weeks + hold_weeks 週已結束的股價；缺少 5 份 csv 的股票會跳過，不會觸發 DataScrape
    每個大戶成本 (ROUND_TRIP_COSTS) 另外算：
    - f_價差 = 散戶成本 / 大戶成本 - 1 (兩邊都照成本價成交時的獲利)
    - f_需要漲幅 = 散戶成本 / min(大戶成本, 錨點收盤) - 1 (從預計的買價要漲多少才賣得到)
    - f_價差波動比 = f_價差 / f_週波動率 (價差相當於幾週的平均波動)
    - f_收盤位置 = (錨點收盤 - 大戶成本) / (散戶成本 - 大戶成本)：< 0 低於大戶成本，0 ~ 1 在兩個成本之間，> 1 高於散戶成本"""
    data_dir = Path(data_dir or DATA_DIR)
    rows, skipped = [], 0
    for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        abbr = folder.name
        data = load_stock_week_data(abbr, data_dir)
        if data is None:
            skipped += 1
            continue
        price = data[ds.PRICE_FILE_TAG]
        weeks = {tag: list(df["年週"]) for tag, df in data.items()}
        current_week = _first_unfinished_week(_latest_week_file(folder, abbr, ds.PRICE_FILE_TAG))
        for pivot in weeks[MARGIN_FILE_TAG]:
            if not all(pivot in w and w.index(pivot) + input_week_num <= len(w) for w in weeks.values()):
                continue
            p = weeks[ds.PRICE_FILE_TAG].index(pivot)
            future = price.iloc[max(p - buy_weeks - hold_weeks, 0) : p].iloc[::-1]  # 錨點之後的週 (由舊到新)
            if len(future) < buy_weeks + hold_weeks or future["年週"].iloc[-1] >= current_week:
                continue
            try:
                big = _institutional_avg_cost(data[INSTITUTIONAL_FILE_TAG], price, pivot, input_week_num)
                margin = _week_window(data[MARGIN_FILE_TAG], pivot, input_week_num, "資券餘額")
                retail = _buy_avg_cost(margin, margin[MARGIN_CHANGE_COLUMN], _week_typical_price(price, margin), "融資")
                features = round_trip_features(data, pivot, input_week_num)
            except ValueError:  # 這個錨點的資料有問題 (例如某週沒有股價)，只跳過這個樣本
                continue
            close = price["收"].iloc[p]
            row = {"股票": abbr, "錨點": pivot, "錨點收盤": close, "成本_散戶": retail, **features}
            for name, group in ROUND_TRIP_COSTS.items():
                cost = big[group]
                ok = not np.isnan(cost) and not np.isnan(retail) and cost < retail
                bought, sold, buy_at, sell_at, buy_week, sell_week = (
                    simulate_round_trip(future, cost, retail, buy_weeks, hold_weeks) if ok
                    else (False, False, np.nan, np.nan, np.nan, np.nan)
                )
                spread = retail / cost - 1
                row.update({
                    f"成本_{name}": cost,
                    f"f_距{name}成本": close / cost - 1,
                    f"f_價差_{name}": spread,
                    f"f_需要漲幅_{name}": retail / np.minimum(cost, close) - 1,
                    f"f_價差波動比_{name}": spread / row["f_週波動率"] if row["f_週波動率"] else np.nan,
                    f"f_收盤位置_{name}": (close - cost) / (retail - cost) if ok else np.nan,
                    f"前提_{name}": ok, f"買到_{name}": bought, f"賣到_{name}": sold,
                    f"報酬_{name}": sell_at / buy_at - 1 if bought else np.nan,
                    f"持有週數_{name}": sell_week - buy_week,
                    f"出場週_{name}": sell_week,
                })
            rows.append(row)
    print(f"來回交易回測：股票 {len({r['股票'] for r in rows})} 檔，樣本 {len(rows)} 筆；跳過資料不完整的股票 {skipped} 檔")
    return pd.DataFrame(rows)


# ===== 提高來回成功率的模式 (門檻來自 DataCollection 的分析，INPUT_WEEK_NUM = 12、OUTPUT_WEEK_NUM = 4) =====
ROUND_TRIP_MIN_SPREAD = 0.03  # 價差 (散戶成本 / 大戶成本 - 1) 至少 3%：成功時至少賺 3%
MIN_WEEK_SAMPLES = 5  # 「最差一週來回成功率」只看符合樣本數至少這麼多的錨點週


def _near_cost(x, name, max_ratio, low, high):
    """價差不超過 max_ratio 週的平均波動，且錨點收盤價在大戶成本的 low ~ high 之間 (例如 -3% ~ +5%)"""
    distance = x[f"f_距{name}成本"]
    return (x[f"f_價差波動比_{name}"] <= max_ratio) & (distance >= low) & (distance <= high)


# 模式名稱 -> (df, 大戶成本名稱) 回傳是否符合
ROUND_TRIP_PATTERNS = {
    "原本訊號 (大戶籌碼↑ 借券↓ 融資↓)": lambda x, name: (x["f_大戶籌碼"] > 0) & (x["f_借券賣出餘額"] < 0) & (x["f_融資餘額"] < 0),
    "寬鬆：價差 <= 0.5 週波動、距大戶成本 -5% ~ +5%": lambda x, name: _near_cost(x, name, 0.5, -0.05, 0.05),
    "平衡：價差 <= 0.4 週波動、距大戶成本 -3% ~ +5%": lambda x, name: _near_cost(x, name, 0.4, -0.03, 0.05),
    "嚴格：價差 <= 0.3 週波動、距大戶成本 -3% ~ +5%": lambda x, name: _near_cost(x, name, 0.3, -0.03, 0.05),
    "平衡 + 外資持股沒有增加": lambda x, name: _near_cost(x, name, 0.4, -0.03, 0.05) & (x["f_外資持股"] <= 0),
    "嚴格 + 外資持股沒有增加": lambda x, name: _near_cost(x, name, 0.3, -0.03, 0.05) & (x["f_外資持股"] <= 0),
}


def _round_trip_stats(x, name, n_weeks, early_weeks):
    """一組樣本的來回交易統計 (定義見「來回交易回測」說明)；n_weeks 是全部的錨點週數，early_weeks 是前期的錨點週"""
    sold, bought = x[f"賣到_{name}"], x[f"買到_{name}"]
    returns = x[f"報酬_{name}"]
    early = x["錨點"].isin(early_weeks)
    by_week = x.groupby("錨點")[f"賣到_{name}"].agg(["mean", "size"])
    return {
        "機會數": len(x),
        "平均每週機會數": len(x) / n_weeks,
        "價差中位數": x[f"f_價差_{name}"].median(),
        "買到率": bought.mean(),
        "買到後成功率": sold[bought].mean(),
        "來回成功率": sold.mean(),
        "前期來回成功率": sold[early].mean(),
        "後期來回成功率": sold[~early].mean(),
        "最差一週來回成功率": by_week.loc[by_week["size"] >= MIN_WEEK_SAMPLES, "mean"].min(),
        "成功時平均獲利": returns[sold].mean(),
        "未成功時平均報酬": returns[bought & ~sold].mean(),
        "買到後平均報酬": returns[bought].mean(),
        "成功平均持有週數": x.loc[sold, f"持有週數_{name}"].mean(),
    }


def summarize_round_trip(results, name):
    """比較各模式與「基準」(大戶成本 < 散戶成本、價差 >= ROUND_TRIP_MIN_SPREAD 的全部機會) 的來回交易結果"""
    d = results[results[f"前提_{name}"] & (results[f"f_價差_{name}"] >= ROUND_TRIP_MIN_SPREAD)]
    weeks = sorted(d["錨點"].unique())
    early_weeks = weeks[: len(weeks) // 2]  # 前一半的錨點週為前期，其餘為後期
    rows = [{"模式": f"基準：價差 >= {ROUND_TRIP_MIN_SPREAD:.0%} 的全部機會", **_round_trip_stats(d, name, len(weeks), early_weeks)}]
    for pattern, rule in ROUND_TRIP_PATTERNS.items():
        rows.append({"模式": pattern, **_round_trip_stats(d[rule(d, name).fillna(False)], name, len(weeks), early_weeks)})
    return pd.DataFrame(rows).set_index("模式")

### 輸入公司代碼，檢查週資料(5份csv)

In [5]:
# ===== 執行：輸入代碼 → 檢查 ABBR 資料夾的 5 份 CSV (沒有或有誤就用 DataScrape 重新取得) =====
# 重新取得時想看瀏覽器操作過程，可改用 ds.fetch_all_week_data(ABBR, headless=False)
ABBR = ds.ask_stock_code()
ABBR_DIR = ensure_week_data(ABBR)

請輸入台股代碼(4位數字)：2330
✔ 資料夾「2330」內的 5 份 CSV 檔都正確


### 分析參數
* `PIVOT_YEAR_WEEK`：選定的年與該年的第幾週，型別為 `(年, 第幾週)` 的 tuple，採 ISO 週次 (週一到週日為一週)，例如 `(2026, 45)` 是 2026/11/2 ~ 11/8 那一週；某天屬於第幾週可用 `date(2026, 11, 6).isocalendar()[:2]` 查
* `INPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為終點(包含)，往前幾週的數據做數值計算：計算大戶與散戶的平均成本，並用 5 份 csv 中同一段期間的資料做出預測
* `OUTPUT_WEEK_NUM`：以 `PIVOT_YEAR_WEEK` 為起點(不包含)，往後幾週的股價，用來驗證是否照預測的方向變化；`PIVOT_YEAR_WEEK` 之後要有足夠的週資料才能驗證

In [ ]:
PIVOT_YEAR_WEEK = (2026, 30)  # 選定的年與該年的第幾週 (ISO 年, 第幾週)
INPUT_WEEK_NUM = 12           # 以 PIVOT_YEAR_WEEK 為終點(包含)，往前幾週的數據做計算 (大戶/散戶平均成本等)
OUTPUT_WEEK_NUM = 4           # 以 PIVOT_YEAR_WEEK 為起點(不包含)，往後幾週的股價用來驗證預測的變化

### 大戶買入平均成本

In [ ]:
# ===== 執行：計算大戶買入平均成本 =====
avg_cost = calc_institutional_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
for name, cost in avg_cost.items():
    print(f"{ABBR} {name} 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{cost:.2f}")

### 散戶買入平均成本

In [ ]:
# ===== 執行：計算散戶買入平均成本 =====
retail_cost = calc_retail_avg_cost(ABBR, PIVOT_YEAR_WEEK, INPUT_WEEK_NUM)
print(f"{ABBR} 散戶(融資) 平均成本 ({PIVOT_YEAR_WEEK} 往前 {INPUT_WEEK_NUM} 週)：{retail_cost:.2f}")

### 假設驗證(回測)：買進訊號 → 未來股價與大戶平均成本的關係
用 `DataCollectionTest` 中所有 5 份 csv 齊全的股票，對每個 `INPUT_WEEK_NUM` (12 ~ 可用上限) 與每個可用的錨點回測，命中有兩種定義 (`HIT_RULES`)：
* `touch`：未來 `OUTPUT_WEEK_NUM` 週的最低價 <= 大戶成本 (有機會用 <= 成本的價格買進)
* `hold`：未來 `OUTPUT_WEEK_NUM` 週的最低價 >= 大戶成本 (股價守住大戶成本，成本是支撐)

看「同錨點同距離差」是否 > 0：在同一段時間、股價離成本差不多遠的情況下，有訊號的命中率比沒訊號的高，才代表訊號真的有用。

In [ ]:
# ===== 執行：假設驗證 (回測) =====
# 只讀取 DataCollectionTest 中現有的 csv，不會觸發 DataScrape；股票越多越久 (約 1300 檔時約需 6 分鐘)
BACKTEST_DATA_DIR = BASE_DIR / "DataCollectionTest"
# 資券餘額、借券資料只有 30 週，還要留 OUTPUT_WEEK_NUM 週驗證，所以 INPUT_WEEK_NUM 目前最多約 25
BACKTEST_INPUT_WEEK_NUMS = range(12, 26)

backtest_results = backtest_buy_signal(BACKTEST_INPUT_WEEK_NUMS, OUTPUT_WEEK_NUM, BACKTEST_DATA_DIR)
# 回測結果與命中的定義無關，換定義只要重新彙整，不用重跑回測
for hit_rule, (description, _) in HIT_RULES.items():
    print(f"\n命中的定義 ({hit_rule})：{description}")
    display(summarize_backtest(backtest_results, hit_rule).round(3).T)

### 來回交易回測：提高來回成功率 (價差 >= 3%)
用 `DataCollection` 中 5 份 csv 齊全的股票 (不齊全的跳過，不會觸發 DataScrape)，以 `INPUT_WEEK_NUM` 週計算成本與特徵，買進、賣出各等最多 `OUTPUT_WEEK_NUM` 週。

表格欄位：
* 機會數：符合條件的「股票 × 錨點週」數；平均每週機會數 = 機會數 / 錨點週數
* 價差中位數：散戶成本 / 大戶成本 - 1 的中位數 (兩邊都照成本價成交時的獲利)
* 買到率：錨點後 4 週內股價碰到大戶成本、買得到的比例
* 買到後成功率：買到之後，4 週內又碰到散戶成本、賣得到的比例
* 來回成功率：買到而且賣在散戶成本的比例 (= 買到率 × 買到後成功率)；前期 / 後期是錨點週分成前後兩半各自的結果，最差一週是單一錨點週的最低值
* 成功時平均獲利：賣在散戶成本的交易，賣價 / 買價 - 1 的平均
* 未成功時平均報酬：買到但沒碰到散戶成本、在第 4 週收盤賣出的交易的平均報酬
* 買到後平均報酬：所有買到的交易 (成功 + 未成功) 的平均報酬
* 成功平均持有週數：成功的交易從買進到賣出經過幾週

In [ ]:
# ===== 執行：來回交易回測 (買進 <= 大戶成本、賣出 >= 散戶成本) =====
# 只讀取 DataCollection (DATA_DIR) 中現有的 csv，不會觸發 DataScrape；約 1900 檔股票時約需 1 ~ 2 分鐘
round_trip_results = backtest_round_trip(INPUT_WEEK_NUM, OUTPUT_WEEK_NUM, OUTPUT_WEEK_NUM, DATA_DIR)
for name in ROUND_TRIP_COSTS:
    print(f"\n大戶成本 = {name}")
    display(summarize_round_trip(round_trip_results, name).round(3))